# **META 01** - Julgar mais processos que os distribuídos

# **Importando os dados**


In [ ]:
import pandas as pd
import csv
import gc
import ast


In [ ]:
arquivos = {
    "CN": "CN_TJDFT.csv",
    "CP": "CP_TJDFT.csv",
    "Sent": "Sent_TJDFT.csv",
    "TBaix": "TBaix_TJDFT.csv"
}

# Colunas que vamos utilizar
colunas = {
    "CN": [
        "Grau", "Processo", "Data de inicio",
        "Data de Referencia", "id_procedimento",
        "Recurso", "Recurso Originario"
    ],

    "CP": [
        "Grau", "Processo", "Datas inicio:fim",
        "Liquido"
    ],

    "Sent": [
        "Grau", "Processo",
        "Data do julgamento: Situacao: Movimento",
        "Data da decisao: Situacao: Movimento", "id_procedimento"
    ],

    "TBaix": [
        "Grau", "Processo", "Data de Referencia",
        "id_procedimento"
    ]
}

dados = {}

for nome, arquivo in arquivos.items():

    print("Carregando:", nome)

    caminho = "/content/" + arquivo

    df = pd.read_csv(
        caminho,
        sep=";",
        engine="python",
        quoting=csv.QUOTE_NONE,
        dtype=str,
        encoding="utf-8-sig",
        usecols=lambda c: c.strip().strip('"') in colunas[nome],
        on_bad_lines="warn"
    )

    # Retirar aspas dos nomes das colunas
    df.columns = df.columns.str.strip().str.strip('"')

    dados[nome] = df

    print("Linhas:", len(df))
    print("Colunas:", len(df.columns))

    del df
    gc.collect()

print("Carregamento finalizado!")

Carregando: CN
Linhas: 893063
Colunas: 7
Carregando: CP
Linhas: 670512
Colunas: 4
Carregando: Sent
Linhas: 1007779
Colunas: 5
Carregando: TBaix
Linhas: 980772
Colunas: 4
Carregamento finalizado!


In [ ]:
print(df_sent["id_procedimento"].value_counts(dropna=False))

id_procedimento
2    540979
1    156191
7    129093
3    123896
4     37423
5     12227
6      7970
Name: count, dtype: int64


# **Criar os Dataframes**

In [ ]:
df_cn = dados["CN"]
df_cp = dados["CP"]
df_sent = dados["Sent"]
df_tbaix = dados["TBaix"]

# **Preparar os dados**

In [ ]:
# Período da pesquisa
inicio = pd.Timestamp("2025-12-20")
fim = pd.Timestamp("2026-07-31")

In [ ]:
# Retirar aspas e espaços
for df in [df_cn, df_cp, df_sent, df_tbaix]:
    df.columns = df.columns.str.strip().str.strip('"')

    for coluna in df.columns:
        df[coluna] = df[coluna].str.strip().str.strip('"')

In [ ]:
# Criar datas
df_cn["data"] = pd.to_datetime(df_cn["Data de inicio"], errors="coerce")
df_tbaix["data"] = pd.to_datetime(df_tbaix["Data de Referencia"], errors="coerce")

In [ ]:
# Função simples para criar uma chave de processo
def chave(linha):
    return (linha["Grau"], linha["Processo"])

**Atenção:** o filtro inicial do glossário (id_fase_processual e id_tipo_procedimento) não pode ser aplicado literalmente, porque essas colunas não estão disponíveis. Vamos usar id_procedimento igual a 1 ou 2 como aproximação nos indicadores

# **CNM1 - Casos novos**

In [ ]:
import pandas as pd

inicio = pd.Timestamp("2025-12-20")
fim = pd.Timestamp("2026-07-31")

# Converter corretamente as datas
df_cn["data"] = pd.to_datetime(
    df_cn["Data de inicio"].str.strip().str.strip('"'),
    format="%Y-%m-%d",
    errors="coerce"
)

# Filtrar registros no período e procedimentos 1 ou 2
cnm1 = df_cn.loc[
    df_cn["data"].between(inicio, fim) &
    df_cn["id_procedimento"].str.strip().str.strip('"').isin(["1", "2"]),
    ["Grau", "Processo"]
]

# Contar processos distintos por grau
chaves_cnm1 = set(
    zip(
        cnm1["Grau"].str.strip().str.strip('"'),
        cnm1["Processo"].str.strip().str.strip('"')
    )
)

print("Registros filtrados:", len(cnm1))
print("CNM1 estimado:", len(chaves_cnm1))
print("Registros duplicados:", len(cnm1) - len(chaves_cnm1))

Registros filtrados: 269578
CNM1 estimado: 269578
Registros duplicados: 0


# **PrimeiraSentM1 - Primeira sentença**

verificando o formato

In [ ]:
print(df_sent["Data do julgamento: Situacao: Movimento"].head(5).to_string())

0      [2025-05-30:72:10953, 2026-08-25:27:220]
1      [2025-05-30:72:10953, 2026-08-25:27:220]
2    [2026-08-05:27:11878, 2026-08-05:28:10961]
3                         [2025-09-30:28:10961]
4                         [2025-06-09:28:10961]


Adicionar id_procedimento ao df_sent

In [ ]:
df_sent = dados["Sent"]

df_sent["id_procedimento"] = (
    df_sent["id_procedimento"]
    .str.strip()
    .str.strip('"')
)

print(df_sent["id_procedimento"].value_counts())

id_procedimento
2    540979
1    156191
7    129093
3    123896
4     37423
5     12227
6      7970
Name: count, dtype: int64


In [ ]:
import re
from collections import Counter

padrao = re.compile(r"(\d{4}-\d{2}-\d{2}):(\d+):(\d+)")

situacoes_validas = {"27", "28", "62", "18", "29", "90", "129"}

primeira_data = {}

for grau, processo, procedimento, texto in df_sent[
    ["Grau", "Processo", "id_procedimento",
     "Data do julgamento: Situacao: Movimento"]
].itertuples(index=False, name=None):

    grau = str(grau).strip().strip('"')
    processo = str(processo).strip().strip('"')
    procedimento = str(procedimento).strip().strip('"')

    if procedimento not in {"1", "2"}:
        continue

    if not isinstance(texto, str):
        continue

    chave = (grau, processo)

    for data, situacao, movimento in padrao.findall(texto):

        if situacao not in situacoes_validas:
            continue

        if chave not in primeira_data or data < primeira_data[chave]:
            primeira_data[chave] = data

chaves_primeirasent = set()

for chave, data in primeira_data.items():
    if "2025-12-20" <= data <= "2026-08-31":
        chaves_primeirasent.add(chave)

print("PrimeiraSentM1 estimado:", len(chaves_primeirasent))

resultado = Counter()

for grau, processo in chaves_primeirasent:
    resultado[grau] += 1

print("\nPor grau:")
for grau, quantidade in sorted(resultado.items()):
    print(grau, quantidade)

PrimeiraSentM1 estimado: 250040

Por grau:
G1 90939
G2 57307
JE 88693
TR 13101


# **DecM1 - Decisões**

In [ ]:
import re
from collections import Counter

padrao = re.compile(r"(\d{4}-\d{2}-\d{2}):(\d+):(\d+)")

primeira_data_dec = {}

colunas = [
    "Grau",
    "Processo",
    "id_procedimento",
    "Data da decisao: Situacao: Movimento"
]

for grau, processo, procedimento, texto in df_sent[colunas].itertuples(
    index=False, name=None
):

    grau = str(grau).strip().strip('"')
    processo = str(processo).strip().strip('"')
    procedimento = str(procedimento).strip().strip('"')

    if procedimento not in {"1", "2"}:
        continue

    if not isinstance(texto, str):
        continue

    chave = (grau, processo)

    for data, situacao, movimento in padrao.findall(texto):

        if situacao != "140":
            continue

        if chave not in primeira_data_dec or data < primeira_data_dec[chave]:
            primeira_data_dec[chave] = data

# Contar somente a primeira ocorrência no período
chaves_dec = set()

for chave, data in primeira_data_dec.items():

    if "2025-12-20" <= data <= "2026-07-31":
        chaves_dec.add(chave)

print("DecM1 - candidatos:", len(chaves_dec))

# Resultado por grau
resultado_dec = Counter()

for grau, processo in chaves_dec:
    resultado_dec[grau] += 1

print("\nResultado por grau:")

for grau, quantidade in sorted(resultado_dec.items()):
    print(grau, quantidade)

DecM1 - candidatos: 447

Resultado por grau:
G1 413
G2 18
JE 16


# **BaixM1 - Processos baixados**

In [ ]:
inicio = pd.Timestamp("2025-12-20")
fim = pd.Timestamp("2026-07-31")

# Corrigir a conversão das datas
df_tbaix["data"] = pd.to_datetime(
    df_tbaix["Data de Referencia"].str.strip().str.strip('"'),
    format="%Y-%m-%d",
    errors="coerce"
)

# Filtrar os registros
baixm1 = df_tbaix.loc[
    df_tbaix["data"].between(inicio, fim) &
    df_tbaix["id_procedimento"].str.strip().str.strip('"').isin(["1", "2"]),
    ["Grau", "Processo"]
]

# Criar chaves únicas
chaves_baix = set(
    zip(
        baixm1["Grau"].str.strip().str.strip('"'),
        baixm1["Processo"].str.strip().str.strip('"')
    )
)

print("Registros filtrados:", len(baixm1))
print("BaixM1 estimado:", len(chaves_baix))
print("Registros duplicados:", len(baixm1) - len(chaves_baix))

Registros filtrados: 258833
BaixM1 estimado: 258832
Registros duplicados: 1


# **MPUM1 - Medidas protetivas de urgência**

In [ ]:
import re

padrao = re.compile(r"(\d{4}-\d{2}-\d{2}):(\d+):(\d+)")

movimentos_mpu = {
    "11423", "11424", "11425", "11426",
    "12476", "12478", "12479",
    "14681", "14733", "15587"
}

movimentos_mpu.update(str(n) for n in range(15486, 15511))

primeira_data_mpu = {}

for grau, processo, texto in df_sent[
    ["Grau", "Processo", "Data da decisao: Situacao: Movimento"]
].itertuples(index=False, name=None):

    if not isinstance(texto, str):
        continue

    chave = (
        str(grau).strip().strip('"'),
        str(processo).strip().strip('"')
    )

    for data, situacao, movimento in padrao.findall(texto):

        if situacao not in {"17", "34"}:
            continue

        if movimento not in movimentos_mpu:
            continue

        if chave not in primeira_data_mpu or data < primeira_data_mpu[chave]:
            primeira_data_mpu[chave] = data

chaves_mpu = {
    chave
    for chave, data in primeira_data_mpu.items()
    if "2025-12-20" <= data <= "2026-07-31"
}

print("MPUM1 - candidatos:", len(chaves_mpu))

MPUM1 - candidatos: 1059


# **JulgadoM1 - Processos julgados**

In [ ]:
chaves_mpu_12 = set()

for grau, processo, procedimento in df_sent[
    ["Grau", "Processo", "id_procedimento"]
].itertuples(index=False, name=None):

    chave = (
        str(grau).strip().strip('"'),
        str(processo).strip().strip('"')
    )

    procedimento = str(procedimento).strip().strip('"')

    if chave in chaves_mpu and procedimento in {"1", "2"}:
        chaves_mpu_12.add(chave)

print("MPUM1 sem filtro:", len(chaves_mpu))
print("Candidatos presentes nos procedimentos 1 ou 2:", len(chaves_mpu_12))
print("Candidatos fora dos procedimentos 1 ou 2:", len(chaves_mpu - chaves_mpu_12))

MPUM1 sem filtro: 1059
Candidatos presentes nos procedimentos 1 ou 2: 1056
Candidatos fora dos procedimentos 1 ou 2: 3


In [ ]:
primeira_data_mpu_12 = {}

for grau, processo, procedimento, texto in df_sent[
    ["Grau", "Processo", "id_procedimento",
     "Data da decisao: Situacao: Movimento"]
].itertuples(index=False, name=None):

    procedimento = str(procedimento).strip().strip('"')

    if procedimento not in {"1", "2"} or not isinstance(texto, str):
        continue

    chave = (
        str(grau).strip().strip('"'),
        str(processo).strip().strip('"')
    )

    for data, situacao, movimento in padrao.findall(texto):

        if situacao not in {"17", "34"}:
            continue

        if movimento not in movimentos_mpu:
            continue

        if chave not in primeira_data_mpu_12 or data < primeira_data_mpu_12[chave]:
            primeira_data_mpu_12[chave] = data

chaves_mpu_12_eventos = {
    chave
    for chave, data in primeira_data_mpu_12.items()
    if "2025-12-20" <= data <= "2026-07-31"
}

print("MPUM1 estimado com filtro no evento:", len(chaves_mpu_12_eventos))

MPUM1 estimado com filtro no evento: 1056


# **DesM1 - Processos dessobrestados**

In [ ]:
import re

chaves_des = set()

# Identifica intervalos no formato 20200303:20230815
padrao_cp = re.compile(r"(\d{8}):(\d{8}|0)")

for grau, processo, texto in df_cp[
    ["Grau", "Processo", "Datas inicio:fim"]
].itertuples(index=False, name=None):

    if not isinstance(texto, str):
        continue

    chave_processo = (
        str(grau).strip().strip('"'),
        str(processo).strip().strip('"')
    )

    intervalos = []

    for data_inicio, data_fim in padrao_cp.findall(texto):

        abertura = pd.to_datetime(
            data_inicio, format="%Y%m%d", errors="coerce"
        )

        encerramento = pd.to_datetime(
            data_fim, format="%Y%m%d", errors="coerce"
        )

        if pd.notna(abertura):
            intervalos.append((abertura, encerramento))

    intervalos.sort(key=lambda x: x[0])

    for i in range(1, len(intervalos)):

        abertura_atual = intervalos[i][0]

        if inicio <= abertura_atual <= fim:

            for anterior in intervalos[:i]:

                if (
                    pd.notna(anterior[1])
                    and anterior[1] < abertura_atual
                ):
                    chaves_des.add(chave_processo)
                    break

            if chave_processo in chaves_des:
                break

print("DesM1 - candidatos:", len(chaves_des))

DesM1 - candidatos: 72619


# **SusM1 - Processos suspensos**

In [ ]:
chaves_julgados = (
    chaves_primeirasent
    | chaves_baix
    | chaves_dec
    | chaves_mpu
)

print("JulgadoM1 estimado:", len(chaves_julgados))

JulgadoM1 estimado: 333910


In [ ]:
inicio = pd.Timestamp("2025-12-20")
fim = pd.Timestamp("2026-07-31")

# Identificar intervalos no formato AAAAMMDD:AAAAMMDD
# O valor 0 indica ausência de data de encerramento
padrao_cp = re.compile(r"(\d{8}):(\d{8}|0)")

chaves_suspensos = set()

for grau, processo, liquido, texto in df_cp[
    ["Grau", "Processo", "Liquido", "Datas inicio:fim"]
].itertuples(index=False, name=None):

    liquido = str(liquido).strip().strip('"')

    if liquido != "N":
        continue

    if not isinstance(texto, str):
        continue

    chave_processo = (
        str(grau).strip().strip('"'),
        str(processo).strip().strip('"')
    )

    datas_encerramento = []

    for data_inicio, data_fim in padrao_cp.findall(texto):

        if data_fim == "0":
            continue

        encerramento = pd.to_datetime(
            data_fim,
            format="%Y%m%d",
            errors="coerce"
        )

        if pd.notna(encerramento):
            datas_encerramento.append(encerramento)

    if not datas_encerramento:
        continue

    # Considerar a última data de encerramento registrada
    ultimo_fim = max(datas_encerramento)

    if inicio <= ultimo_fim <= fim:
        chaves_suspensos.add(chave_processo)

# Processos pertencentes ao CNM1 ou DesM1
universo = chaves_cnm1 | chaves_des

# Excluir os processos presentes no JulgadoM1
chaves_sus = (chaves_suspensos & universo) - chaves_julgados

print("Candidatos à suspensão:", len(chaves_suspensos))
print("SusM1 estimado:", len(chaves_sus))

Candidatos à suspensão: 40721
SusM1 estimado: 15053


# **Resultado final da Meta 1**

In [ ]:
cnm1_total = len(chaves_cnm1)
desm1_total = len(chaves_des)
susm1_total = len(chaves_sus)
julgadom1_total = len(chaves_julgados)

denominador = cnm1_total + desm1_total - susm1_total
meta1 = julgadom1_total / denominador * 100

print("CNM1:", cnm1_total)
print("DesM1:", desm1_total)
print("SusM1:", susm1_total)
print("JulgadoM1:", julgadom1_total)
print("Denominador:", denominador)
print(f"Meta 1 estimada: {meta1:.2f}%")

CNM1: 269578
DesM1: 72619
SusM1: 15053
JulgadoM1: 333910
Denominador: 327144
Meta 1 estimada: 102.07%


# **Meta 02** - Julgar processos mais antigos


Fórmulas para a meta 02

In [ ]:
# Período de apuração
inicio_m2 = "2026-01-01"
fim_m2 = "2026-12-31"

# Datas-limite dos processos antigos
corte_m2 = {
    "a": "2022-12-31",    # 1º grau
    "b": "2023-12-31",    # 2º grau
    "c": "2023-12-31",    # Juizados e Turmas
    "ant": "2011-12-31"   # 15 anos ou mais
}

# Percentuais exigidos
meta_exigida = {
    "a": 80,
    "b": 90,
    "c": 95,
    "ant": 100
}

def calcular_meta2(julgados, distribuidos, suspensos, dessobrestados=0, grupo="a"):

    if grupo == "ant":
        denominador = distribuidos - suspensos - dessobrestados
    else:
        denominador = distribuidos - suspensos

    if denominador <= 0:
        return None

    percentual_julgado = julgados / denominador * 100
    cumprimento = percentual_julgado / meta_exigida[grupo] * 100

    return round(cumprimento, 2)